# Edge and Region Lab

This notebook reorganizes the `#4 엣지와 영역` lecture into a compact lab.

Topics:
- Sobel, Canny, and LoG edge detection
- contour extraction
- Hough line and circle detection
- SLIC superpixels
- region features

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage import data, segmentation

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = Path("lab-session/03-edge-and-region")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

DOG_PATH = DATA_DIR / "dog1.jpg"
CHESS_PATH = DATA_DIR / "chess.png"
APPLE_PATH = DATA_DIR / "apple.jpg"
print("Base dir:", BASE_DIR.resolve())

## Edge detection

In [ ]:
gray = cv2.imread(str(DOG_PATH), cv2.IMREAD_GRAYSCALE)
if gray is None:
    raise FileNotFoundError(DOG_PATH)

grad_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
grad_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
sobel_x = cv2.convertScaleAbs(grad_x)
sobel_y = cv2.convertScaleAbs(grad_y)
sobel_mag = cv2.addWeighted(sobel_x, 0.5, sobel_y, 0.5, 0)
canny = cv2.Canny(gray, 100, 200)
log = cv2.Laplacian(cv2.GaussianBlur(gray, (3, 3), 0.1), cv2.CV_64F)
log = np.uint8(np.absolute(log))

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, image, title in zip(
    axes,
    [gray, sobel_mag, canny, log],
    ["Gray", "Sobel magnitude", "Canny", "LoG"],
):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## Contours from the edge map

In [ ]:
dog_bgr = cv2.imread(str(DOG_PATH))
contours, _ = cv2.findContours(canny, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
long_contours = [cnt for cnt in contours if cnt.shape[0] > 100]
contour_view = dog_bgr.copy()
cv2.drawContours(contour_view, long_contours, -1, (0, 255, 0), 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(cv2.cvtColor(dog_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(canny, cmap="gray")
axes[1].set_title("Canny edge map")
axes[2].imshow(cv2.cvtColor(contour_view, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Contours ({len(long_contours)})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Hough line detection

In [ ]:
chess = cv2.imread(str(CHESS_PATH))
if chess is None:
    raise FileNotFoundError(CHESS_PATH)
chess_gray = cv2.cvtColor(chess, cv2.COLOR_BGR2GRAY)
chess_edges = cv2.Canny(chess_gray, 100, 200)
lines = cv2.HoughLines(chess_edges, 1, np.pi / 180, 130)
line_overlay = chess.copy()
line_count = 0
if lines is not None:
    for line in lines:
        rho, theta = line[0]
        a = np.cos(theta)
        b = np.sin(theta)
        x0 = a * rho
        y0 = b * rho
        x1 = int(x0 + 1000 * (-b))
        y1 = int(y0 + 1000 * a)
        x2 = int(x0 - 1000 * (-b))
        y2 = int(y0 - 1000 * a)
        cv2.line(line_overlay, (x1, y1), (x2, y2), (255, 0, 0), 2)
        line_count += 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(cv2.cvtColor(chess, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(chess_edges, cmap="gray")
axes[1].set_title("Edge map")
axes[2].imshow(cv2.cvtColor(line_overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Hough lines ({line_count})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Hough circle detection

In [ ]:
apple = cv2.imread(str(APPLE_PATH))
if apple is None:
    raise FileNotFoundError(APPLE_PATH)
apple_gray = cv2.cvtColor(apple, cv2.COLOR_BGR2GRAY)
apple_gray = cv2.medianBlur(apple_gray, 5)
circles = cv2.HoughCircles(apple_gray, cv2.HOUGH_GRADIENT, 1, 120, param1=150, param2=20, minRadius=20, maxRadius=140)
circle_overlay = apple.copy()
circle_count = 0
if circles is not None:
    for circle in circles[0]:
        x, y, radius = int(circle[0]), int(circle[1]), int(circle[2])
        cv2.circle(circle_overlay, (x, y), radius, (0, 255, 255), 2)
        circle_count += 1

plt.figure(figsize=(6, 4))
plt.imshow(cv2.cvtColor(circle_overlay, cv2.COLOR_BGR2RGB))
plt.title(f"Hough circles ({circle_count})")
plt.axis("off");

## SLIC superpixels

In [ ]:
coffee = data.coffee()
slic_a = segmentation.slic(coffee, compactness=20, n_segments=400, start_label=1)
slic_b = segmentation.slic(coffee, compactness=40, n_segments=400, start_label=1)
marked_a = segmentation.mark_boundaries(coffee, slic_a)
marked_b = segmentation.mark_boundaries(coffee, slic_b)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, image, title in zip(
    axes,
    [coffee, marked_a, marked_b],
    ["Original", "SLIC compactness=20", "SLIC compactness=40"],
):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## Region features

In [ ]:
horse = data.horse()
horse_image = 255 - np.uint8(horse) * 255
horse_contours, _ = cv2.findContours(horse_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
horse_contour = horse_contours[0]

moments = cv2.moments(horse_contour)
area = cv2.contourArea(horse_contour)
perimeter = cv2.arcLength(horse_contour, True)
roundness = (4.0 * np.pi * area) / (perimeter * perimeter)
center_x = moments["m10"] / moments["m00"]
center_y = moments["m01"] / moments["m00"]

feature_view = cv2.cvtColor(horse_image, cv2.COLOR_GRAY2BGR)
approx = cv2.approxPolyDP(horse_contour, 8, True)
hull = cv2.convexHull(horse_contour).reshape(1, -1, 2)
cv2.drawContours(feature_view, [horse_contour], -1, (255, 0, 255), 2)
cv2.drawContours(feature_view, [approx], -1, (0, 255, 0), 2)
cv2.drawContours(feature_view, hull, -1, (0, 0, 255), 2)
cv2.circle(feature_view, (int(center_x), int(center_y)), 4, (255, 255, 0), -1)

plt.figure(figsize=(7, 5))
plt.imshow(cv2.cvtColor(feature_view, cv2.COLOR_BGR2RGB))
plt.title(f"Area={area:.1f}, Perimeter={perimeter:.1f}, Roundness={roundness:.4f}")
plt.axis("off")
print(f"Center: ({center_x:.1f}, {center_y:.1f})")

## What to run next

Run the window-based demos from this folder when you want the full interactive view:
- `python examples/01_sobel_canny_log.py`
- `python examples/02_contour_detection.py`
- `python examples/03_hough_lines.py`
- `python examples/04_hough_circles.py`
- `python examples/05_superpixels_slic.py`
- `python examples/06_region_features.py`